<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic%20%7C%20L8%20-%20Recurrent%20Neural%20Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8: Recurrent Neural Networks (RNNs)

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** 2024  
**Updated:** 2024

## Learning Objectives
- Understand sequence modeling and temporal dependencies
- Learn RNN architecture and its limitations
- Master LSTM and GRU architectures
- Build models for text generation and sentiment analysis
- Understand bidirectional RNNs

## Prerequisites
- Completed L1-L7
- Understanding of neural networks and backpropagation

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import string

print(f"TensorFlow version: {tf.__version__}")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

## 1. Why RNNs for Sequential Data?

### Sequential Data Examples:
- **Text:** "The cat sat on the ___" (next word depends on previous words)
- **Time Series:** Stock prices, weather data
- **Speech:** Audio signals
- **Video:** Sequence of frames

### Problems with Regular Neural Networks:
1. **Fixed input size:** Can't handle variable-length sequences
2. **No memory:** Each input processed independently
3. **No temporal understanding:** Order doesn't matter

### RNN Solution:
- **Hidden state** acts as memory
- **Recurrent connections** pass information through time
- **Variable length** sequences supported

## 2. Understanding RNN Architecture

### Basic RNN:
```
h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
y_t = W_y * h_t + b_y
```

Where:
- `h_t`: Hidden state at time t
- `x_t`: Input at time t
- `y_t`: Output at time t

### Problem: Vanishing Gradients
- Long sequences → gradients become very small
- Network can't learn long-term dependencies
- **Solution:** LSTM and GRU

In [ ]:
# Visualize RNN unrolling
from IPython.display import Image, display

print("RNN Unrolled Through Time:")
print("""\n    x_1      x_2      x_3      x_4
     ↓        ↓        ↓        ↓
   [RNN] → [RNN] → [RNN] → [RNN]
     ↓        ↓        ↓        ↓
    y_1      y_2      y_3      y_4
    
Each RNN cell:
- Takes input x_t
- Receives hidden state from previous step
- Produces output y_t
- Passes hidden state to next step
""")

## 3. LSTM (Long Short-Term Memory)

LSTM solves vanishing gradient problem with:
- **Cell state:** Long-term memory
- **Gates:** Control information flow
  - **Forget gate:** What to forget from cell state
  - **Input gate:** What new information to add
  - **Output gate:** What to output

### LSTM Equations:
```
f_t = σ(W_f * [h_{t-1}, x_t] + b_f)  # Forget gate
i_t = σ(W_i * [h_{t-1}, x_t] + b_i)  # Input gate
C̃_t = tanh(W_C * [h_{t-1}, x_t] + b_C)  # Candidate values
C_t = f_t * C_{t-1} + i_t * C̃_t  # New cell state
o_t = σ(W_o * [h_{t-1}, x_t] + b_o)  # Output gate
h_t = o_t * tanh(C_t)  # New hidden state
```

## 4. Text Generation with LSTM

Let's build a character-level text generator!

In [ ]:
# Sample text data
text = """Deep learning is a subset of machine learning in artificial intelligence that has networks capable of learning unsupervised from data that is unstructured or unlabeled. Also known as deep neural learning or deep neural network. Deep learning models are built using neural networks with multiple layers. The depth of the model is represented by the number of layers in the model. Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning."""

print(f"Text length: {len(text)} characters")
print(f"\nFirst 200 characters:\n{text[:200]}")

# Create character mappings
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"\nUnique characters: {len(chars)}")
print(f"Characters: {''.join(chars)}")

In [ ]:
# Prepare training data
seq_length = 40  # Length of input sequences
step = 3  # Step size for creating sequences

sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i:i + seq_length])
    next_chars.append(text[i + seq_length])

print(f"Number of sequences: {len(sentences)}")
print(f"\nExample sequence:")
print(f"Input: '{sentences[0]}'")
print(f"Target: '{next_chars[0]}'")

In [ ]:
# Vectorize the data
X = np.zeros((len(sentences), seq_length, len(chars)), dtype=bool)
y = np.zeros((len(sentences), len(chars)), dtype=bool)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Build LSTM model for text generation
model_lstm = keras.Sequential([
    layers.LSTM(128, input_shape=(seq_length, len(chars)), return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(128),
    layers.Dropout(0.2),
    layers.Dense(len(chars), activation='softmax')
], name='Text_Generator_LSTM')

model_lstm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_lstm.summary()

In [ ]:
# Train the model
print("Training text generation model...")
history = model_lstm.fit(
    X, y,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate text
def sample(preds, temperature=1.0):
    """Sample an index from a probability array with temperature."""
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(seed_text, length=200, temperature=0.5):
    """Generate text starting from seed_text."""
    generated = seed_text
    sentence = seed_text[-seq_length:]
    
    for i in range(length):
        # Prepare input
        x_pred = np.zeros((1, seq_length, len(chars)))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_to_idx[char]] = 1
        
        # Predict next character
        preds = model_lstm.predict(x_pred, verbose=0)[0]
        next_idx = sample(preds, temperature)
        next_char = idx_to_char[next_idx]
        
        generated += next_char
        sentence = sentence[1:] + next_char
    
    return generated

# Generate text with different temperatures
seed = "Deep learning is a subset of machine "

print("Generated Text (Temperature = 0.2 - Conservative):")
print(generate_text(seed, length=200, temperature=0.2))
print("\n" + "="*80 + "\n")

print("Generated Text (Temperature = 0.5 - Balanced):")
print(generate_text(seed, length=200, temperature=0.5))
print("\n" + "="*80 + "\n")

print("Generated Text (Temperature = 1.0 - Creative):")
print(generate_text(seed, length=200, temperature=1.0))

## 5. Sentiment Analysis with LSTM

Let's classify movie reviews as positive or negative!

In [ ]:
# Load IMDB dataset
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

max_features = 10000  # Vocabulary size
maxlen = 200  # Maximum sequence length

print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(f"Training sequences: {len(x_train)}")
print(f"Test sequences: {len(x_test)}")
print(f"\nExample review (first 10 words): {x_train[0][:10]}")
print(f"Label: {y_train[0]} (1=positive, 0=negative)")

In [ ]:
# Pad sequences to same length
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

In [ ]:
# Build sentiment analysis model
model_sentiment = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1, activation='sigmoid')
], name='Sentiment_Analysis')

model_sentiment.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_sentiment.summary()

In [ ]:
# Train sentiment model
print("Training sentiment analysis model...")
history_sentiment = model_sentiment.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = model_sentiment.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

## 6. GRU (Gated Recurrent Unit)

GRU is a simpler alternative to LSTM:
- **Fewer parameters** (faster training)
- **Two gates** instead of three (update and reset)
- Often performs similarly to LSTM

In [ ]:
# Build GRU model for comparison
model_gru = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.GRU(64, return_sequences=True),
    layers.GRU(32),
    layers.Dense(1, activation='sigmoid')
], name='Sentiment_GRU')

model_gru.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("GRU Model:")
model_gru.summary()

print("\nLSTM vs GRU Parameters:")
print(f"LSTM: {model_sentiment.count_params():,}")
print(f"GRU: {model_gru.count_params():,}")
print(f"Difference: {model_sentiment.count_params() - model_gru.count_params():,} fewer parameters")

## 7. Bidirectional RNNs

Process sequences in both directions:
- **Forward:** Left to right
- **Backward:** Right to left
- Useful when future context matters (e.g., "I am ___" vs "___ am I")

In [ ]:
# Build bidirectional LSTM
model_bidirectional = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(1, activation='sigmoid')
], name='Bidirectional_LSTM')

model_bidirectional.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_bidirectional.summary()

In [ ]:
# Train bidirectional model
print("Training bidirectional LSTM...")
history_bi = model_bidirectional.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = model_bidirectional.evaluate(x_test, y_test)
print(f"\nBidirectional LSTM Test Accuracy: {test_acc*100:.2f}%")

## 8. Key Takeaways

### RNN Basics:
1. **RNNs** process sequential data with memory
2. **Hidden state** carries information through time
3. **Vanishing gradients** limit basic RNN effectiveness

### LSTM & GRU:
4. **LSTM** uses gates to control information flow
5. **GRU** is simpler with fewer parameters
6. Both solve vanishing gradient problem

### Applications:
7. **Text generation:** Character/word-level prediction
8. **Sentiment analysis:** Classify text sentiment
9. **Bidirectional RNNs:** Use future context

### Best Practices:
10. Use **LSTM/GRU** instead of basic RNN
11. Add **dropout** to prevent overfitting
12. Consider **bidirectional** for classification tasks

## Next Steps (L9)
- Learn about Attention Mechanisms
- Understand Transformer architecture
- Explore self-attention and multi-head attention

## References
- Understanding LSTM Networks: http://colah.github.io/posts/2015-08-Understanding-LSTMs/
- The Unreasonable Effectiveness of RNNs: http://karpathy.github.io/2015/05/21/rnn-effectiveness/
- TensorFlow RNN Tutorial: https://www.tensorflow.org/guide/keras/rnn